# RHAPSODY Dask Backend Tutorial

This tutorial shows how to use the **`DaskExecutionBackend`** to run tasks on Dask clusters.
The backend is **cluster-agnostic**: the same task code works unchanged across:

| Section | Cluster type |
|---------|-------------|
| 1–3 | `LocalCluster` (zero-config, laptop / single node) |
| 4 | `SLURMCluster` (HPC / SLURM) |
| 5 | `KubeCluster` (Kubernetes) |
| 6 | GPU resource-aware scheduling |

Sections 4–6 show reference code that requires the respective infrastructure to run.

In [ ]:
import asyncio

from rhapsody.api import ComputeTask, Session
from rhapsody.backends import DaskExecutionBackend

---

## 1. Sync Functions on a LocalCluster

`DaskExecutionBackend()` with no arguments starts a `LocalCluster` automatically.
Regular (non-async) Python functions are dispatched natively to Dask workers.

In [ ]:
def compute_square(n):
    """Compute the square of a number."""
    return {"input": n, "result": n * n}

In [ ]:
async def run_sync_functions():
    async with DaskExecutionBackend() as backend:
        session = Session(backends=[backend])
        tasks = [ComputeTask(function=compute_square, args=(i,)) for i in range(20)]
        async with session:
            await session.submit_tasks(tasks)
            await session.wait_tasks(tasks)

        done = sum(1 for t in tasks if t.state == "DONE")
        failed = sum(1 for t in tasks if t.state == "FAILED")
        print(f"Done: {done}, Failed: {failed}")
        for t in tasks[:3]:
            print(f"  {t.uid} -> {t.return_value}")

await run_sync_functions()

**Expected output:**
```
Done: 20, Failed: 0
  task.000001 -> {'input': 0, 'result': 0}
  task.000002 -> {'input': 1, 'result': 1}
  task.000003 -> {'input': 2, 'result': 4}
```

---

## 2. Async Functions

Async functions are wrapped transparently and run inside the Dask worker event loop.
The function name is preserved in the Dask dashboard for easy monitoring.

In [ ]:
async def fetch_data(n):
    """Simulate async I/O work."""
    await asyncio.sleep(0.01)
    return n * 2


async def run_async_functions():
    async with DaskExecutionBackend() as backend:
        session = Session(backends=[backend])
        tasks = [ComputeTask(function=fetch_data, args=(i,)) for i in range(10)]
        async with session:
            await session.submit_tasks(tasks)
            await session.wait_tasks(tasks)

        done = sum(1 for t in tasks if t.state == "DONE")
        print(f"Done: {done}/10")
        for t in tasks[:3]:
            print(f"  {t.uid} -> {t.return_value}")

await run_async_functions()

---

## 3. Executable Tasks

Executable tasks run a shell binary inside the Dask worker via `subprocess`.
Results are available via `task.stdout`, `task.stderr`, and `task.exit_code`.

In [ ]:
async def run_executables():
    async with DaskExecutionBackend() as backend:
        session = Session(backends=[backend])
        tasks = [
            ComputeTask(executable="/bin/echo", arguments=[f"task {i}"])
            for i in range(5)
        ]
        async with session:
            await session.submit_tasks(tasks)
            await session.wait_tasks(tasks)

        for t in tasks:
            print(f"{t.uid} | exit={t.exit_code} | stdout={t.stdout.strip()!r}")

await run_executables()

**Expected output:**
```
task.000001 | exit=0 | stdout='task 0'
task.000002 | exit=0 | stdout='task 1'
...
```

---

## 4. SLURM / HPC Cluster

Pass a pre-configured `SLURMCluster` via the `cluster=` parameter.
**The task code is identical to sections 1–3** — only the cluster changes.

Requires `dask-jobqueue`: `pip install dask-jobqueue`

> Run this on a SLURM login node.

In [ ]:
# from dask_jobqueue import SLURMCluster
#
# cluster = SLURMCluster(
#     cores=4,
#     memory="8GB",
#     walltime="01:00:00",
#     job_extra_directives=["--partition=compute"],
# )
# cluster.scale(jobs=4)  # Submit 4 SLURM jobs as Dask workers
#
# async def run_on_slurm():
#     async with DaskExecutionBackend(cluster=cluster) as backend:
#         session = Session(backends=[backend])
#         tasks = [ComputeTask(function=compute_square, args=(i,)) for i in range(100)]
#         async with session:
#             await session.submit_tasks(tasks)
#             await session.wait_tasks(tasks)
#         done = sum(1 for t in tasks if t.state == "DONE")
#         print(f"{done}/100 tasks completed on SLURM")
#
# asyncio.run(run_on_slurm())

---

## 5. Kubernetes Cluster

Same pattern with `KubeCluster`.

Requires `dask-kubernetes`: `pip install dask-kubernetes`

In [ ]:
# from dask_kubernetes.operator import KubeCluster
#
# cluster = KubeCluster(name="rhapsody-workers", n_workers=4)
#
# async def run_on_k8s():
#     async with DaskExecutionBackend(cluster=cluster) as backend:
#         session = Session(backends=[backend])
#         tasks = [ComputeTask(function=compute_square, args=(i,)) for i in range(50)]
#         async with session:
#             await session.submit_tasks(tasks)
#             await session.wait_tasks(tasks)
#         done = sum(1 for t in tasks if t.state == "DONE")
#         print(f"{done}/50 tasks completed on Kubernetes")
#
# asyncio.run(run_on_k8s())

---

## 6. GPU Resource-Aware Scheduling

Set `gpu=N` on a task to restrict it to Dask workers that advertise GPU resources.
Workers must be started with `--resources "GPU=1"` (or equivalent in the cluster config).

In [ ]:
# async def gpu_task(x):
#     # In real usage: import cupy, torch, etc.
#     return x ** 2
#
# async def run_gpu_tasks():
#     # Workers must advertise GPU resources:
#     #   dask worker <scheduler-address> --resources "GPU=1"
#     async with DaskExecutionBackend() as backend:
#         session = Session(backends=[backend])
#         tasks = [ComputeTask(function=gpu_task, args=(i,), gpu=1) for i in range(4)]
#         async with session:
#             await session.submit_tasks(tasks)
#             await session.wait_tasks(tasks)
#         for t in tasks:
#             print(f"{t.uid} -> {t.return_value}")
#
# await run_gpu_tasks()

---

## Quick Reference

### `DaskExecutionBackend` constructor parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `resources` | `dict` | Passed to `dask.distributed.Client(...)` (e.g. `n_workers`, `threads_per_worker`) |
| `cluster` | Dask cluster object | Pre-configured cluster (`SLURMCluster`, `KubeCluster`, `LocalCluster`, …) |
| `client` | `dask.distributed.Client` | Pre-configured client — takes precedence over `cluster` |
| `name` | `str` | Backend name (default `"dask"`) |

### Task result fields

| Field | Populated by | Description |
|-------|-------------|-------------|
| `task.state` | Session | `"DONE"`, `"FAILED"`, or `"CANCELED"` |
| `task.return_value` | Backend | Return value of the function (function tasks) |
| `task.stdout` | Backend | Captured stdout (executable tasks) |
| `task.stderr` | Backend | Captured stderr (executable tasks) |
| `task.exit_code` | Backend | Process exit code (executable tasks) |
| `task.exception` | Backend | Exception object if task failed |